In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import os
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [3]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset

from sklearn.model_selection import train_test_split
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
data = list(dataset)
kb_data, eval_data = train_test_split(data, test_size=0.2, random_state=42)
eval_data = eval_data[:100]

In [5]:
from langchain_core.documents import Document
documents = []
qa_pairs = []

for item in kb_data:
    qa_pairs_temp = None  # kb_data is only for knowledge base
    documents.append(Document(page_content=item["response"],
                              metadata={"category": item["category"], "intent": item["intent"]}))

for item in eval_data:
    qa_pairs.append({"question": item["instruction"], "answer": item["response"]})

print("Total docs:", len(documents))
print("Total eval pairs:", len(qa_pairs))

Total docs: 21497
Total eval pairs: 100


In [6]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)



In [7]:
def get_combined_context(query, retriever, KG):
    text_docs = retriever.invoke(query)
    text_context = "\n\n".join([d.page_content for d in text_docs])
    if(len(text_docs)==0):
        return None
    
    kg_context = retrieve_kg_context(query, KG)

    combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
    return combined_context


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,     
    chunk_overlap=0,   
    separators=["\n\n", "\n", " ", ""]
)

chunks= text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

# for doc in documents:
#     triples = extract_triples_spacy(doc.page_content)
#     for (subj, rel, obj) in triples:
#         KG.add_triple(subj.strip(), rel.strip(), obj.strip())
from tqdm import tqdm

for doc in tqdm(documents, desc="Building KG"):
    triples = extract_triples_spacy(doc.page_content)
    for (subj, rel, obj) in triples:
        KG.add_triple(subj.strip(), rel.strip(), obj.strip())



Total chunks: 63515


Building KG: 100%|██████████| 21497/21497 [10:06<00:00, 35.46it/s]


In [9]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda:0"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_320691/2493071643.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building

In [10]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 784


index = faiss.IndexFlatIP(dimension)

In [11]:

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(len(chunks))
print(vectorstore.index.ntotal)

63515
63515


In [12]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )

In [21]:
from transformers import pipeline, AutoTokenizer
from langchain_community.llms import HuggingFacePipeline

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    tokenizer=tokenizer,
    device=0,                   # your GPU index
    max_new_tokens=100,          # 3-5 words only
    max_length=None, 
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=pipe)


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2393.17it/s]


In [23]:
def actor(question, context, feedback_section):
    context = context[:1500]
    messages = [
        {"role": "system", "content": (
            "You are a customer support agent. Answer the customer's question "
            "using ONLY the provided context. Be helpful and conversational. "
            "If the answer is not in the context, say 'I cannot help with that.'"
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nCustomer: {question}"}
    ]
    output = pipe(messages)
    return output[0]["generated_text"].strip().split('\n')[0]

In [17]:
# prompt = PromptTemplate(
#     input_variables=["context", "question"],
#     template="""
# You are a factual assistant. Use the following context to answer the question.
# Do NOT add information that is not supported by the context.
# if you don't find the answer in the context then simply say NO.

# Context:
# {context}

# Question: {question}
# Answer:
# """
# )

In [15]:
#evaluator
import json
from langchain_openai import ChatOpenAI



In [16]:
evaluator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}}
)

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a RAG Quality Auditor. You must evaluate ONLY two things:
    1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
    2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
    Output ONLY a JSON object with EXACTLY this structure:
    {{
        "context_relevance": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "faithfulness": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "final_status": "PASS" or "FAIL",
        "action_item": "What should the system do next?(e.g., 'Re-retrieve context' or 'Refine answer').
        if context_relevance score<0.8 then 'Re-retrieve context',if faithfullness score<0.8 then 'Refine answer'.
    }}
    
    Threshold for PASS: Both scores must be >= 0.8.
    """),
    ("user", "USER QUERY: {query}\n\nRETRIEVED CONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
])

In [17]:
def evaluate_response(question,context, answer):
    chain = eval_prompt | evaluator_llm
    # response = chain.invoke({"context": context, "answer": answer})
    
    
    try:
        response = chain.invoke({
            "query": question,
            "context": context,
            "answer": answer
        })
        return json.loads(response.content)
    except Exception as e:
        return {
            "final_status": "FAIL",
            "action_item": f"Error in evaluation: {str(e)}"
        }

In [ ]:

# try:
#     response = evaluator_llm.invoke("Hello!")
#     print("Ollama is awake and responding!")
# except Exception as e:
#     print(f"Connection failed. Is the server running? Error: {e}")

Connection failed. Is the server running? Error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fd1fdde3a90>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [18]:
# query_refiner_llm = ChatOllama(model="qwen2.5:7b", temperature=0)
query_refiner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def generate_refined_query(original_query, critique):
    prompt = f"""
    The initial search for '{original_query}' failed.
    Critique: {critique}
    based on this critique, write a better search query which contains the missing terms or concepts that the original query lacked,
    which led to the failure.
    Don't change the meaning of the question.
    Output ONLY the new search string.
    """
    response = query_refiner_llm.invoke(prompt)
    return response.content.strip()

In [19]:
def run_reflexion_loop(original_question,max_iterations=1):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = " " 
    
    for i in range(max_iterations):
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluate_response(original_question, current_context, answer)


        if evaluation.get('final_status') == "PASS":
            return answer,current_context
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            # print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        else:
            # feedback_section=evaluation['faithfulness']['reason']
            feedback_section = evaluation.get('faithfulness', {}).get('reason', ' ')
    return answer,current_context

In [24]:
import torch
results = []

for i, qa in enumerate(qa_pairs):
    question = qa["question"]
    gt = qa["answer"]
    
    try:
        response, context = run_reflexion_loop(question)
    except RuntimeError as e:
        if "out of memory" in str(e):
            torch.cuda.empty_cache()
            response = "OOM"
            context = ""
        else:
            raise e
    
    results.append({
        "question": question,
        "pred": response,
        "gt": gt,
        "context": context
    })

    torch.cuda.empty_cache()  # unload after each inference
    print(f"q{i}:{response}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


q0:I'm here to help you with any questions or concerns you may have. However, I'm a customer support agent, and it seems like you're trying to interact with a chatbot or a system that's supposed to provide options. If you're having trouble reaching a human agent, I'd be happy to try and assist you further. Can you tell me a bit more about what's going on and how I can help?
q1:I'm happy to help you locate the bills from {{Person Name}}. To provide you with the most accurate information, could you please provide me with some additional details? Specifically, if you could provide me with the date range or any other relevant information, it will greatly assist me in helping you find what you're looking for.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


q2:I cannot help with that.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

q97:I see that we have options to rectify mistakes, and we're very determined to help you with any issues that come up. We also have options to address sign-up errors, so let's explore those options together. Can you tell me a bit more about the error you're experiencing?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


q98:I'd be happy to help you with canceling your order.  We rectify mistakes, and I'm here to support you.  You find options, and we have options to address your issue.  Let's try to resolve this for you.  Can you please confirm your order number, {{Order Number}}, and I'll guide you through the process of canceling it?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


q99:I'd be happy to help you with that. However, I need a bit more information from you. Could you please tell me which article you're interested in ordering and what type of options you're looking for (e.g. digital, physical, etc.)?


In [26]:
right=0
wrong=0
wrong_answers=[]

def normalize(text):
    text = text.strip().lower()
    if text in ["NO", "no answer", "no_answer", "n/a"]:
        return ""
    return text

for i,item in enumerate(results):
    answer=item["pred"].lower()
    gt=item["gt"].lower()

    if normalize(answer) not in normalize(gt):
        wrong+=1
        wrong_answers.append(i)
    else:
        right+=1

print(right)
print(wrong)
print(wrong_answers)



0
100
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [27]:
#ragas
from ragas import EvaluationDataset, evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [28]:
ragas_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
ragas_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [29]:
answerable = []
unanswerable = []

for r in results:
    if not r["gt"] or r["gt"].strip() == "":
        unanswerable.append(r)
    else:
        answerable.append(r)

print(f"Answerable: {len(answerable)}, Unanswerable: {len(unanswerable)}")

# RAGAS only on answerable
dataset_ragas = []
for r in answerable:
    dataset_ragas.append({
        "user_input": r["question"],
        "retrieved_contexts": r["context"] if isinstance(r["context"], list) else [r["context"]],
        "response": r["pred"],
        "reference": r["gt"]
    })

# Separate simple accuracy for unanswerable
# unanswerable_correct = sum(
#     1 for r in unanswerable 
#     if r["pred"].strip().upper() in ["NO", "NO ANSWER", "UNANSWERABLE", "I DON'T KNOW"]
# )
# print(f"Unanswerable accuracy: {unanswerable_correct}/{len(unanswerable)} = {unanswerable_correct/len(unanswerable):.2f}")

Answerable: 100, Unanswerable: 0


In [30]:
evaluation_dataset= EvaluationDataset.from_list(dataset_ragas)

ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    batch_size=4,
)
print(ragas_result)

Evaluating: 100%|██████████| 300/300 [33:38<00:00,  6.73s/it]


{'context_recall': 0.8388, 'faithfulness': 0.5131, 'factual_correctness(mode=f1)': 0.2523}


In [31]:
def normalize(text):
    text = text.strip().lower()
    if text in ["no answers", "no answer", "no_answer", "n/a", "no"]:
        return ""
    return text

# substring match on answerable only
correct = sum(
    normalize(r["gt"]) in normalize(r["pred"])
    for r in answerable
    if normalize(r["gt"]) != ""
)
print(f"Substring Match: {correct}/{len(answerable)} = {correct/len(answerable):.4f}")

# F1 on answerable only
from collections import Counter
def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_em(pred, gt):
    return int(normalize(pred) == normalize(gt))

em_scores=[]
for r in answerable:
    pred = r["pred"]
    gt = r["gt"]
    em_scores.append(compute_em(pred, gt))
exact_mach= sum(em_scores)/len(em_scores)
f1_scores = [compute_f1(r["pred"], r["gt"]) for r in answerable]
print(f"Token F1: {sum(f1_scores)/len(f1_scores):.4f}")
print(f"Exact Match on answerable only: {exact_mach:.4f}")


Substring Match: 0/100 = 0.0000
Token F1: 0.2197
Exact Match on answerable only: 0.0000


In [32]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 215.20it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:

scores = []
skipped = 0
for row in dataset_ragas:
    if not row["reference"] or row["reference"].strip() == "":  # skip empty references
        skipped += 1
        continue
    
    ref_emb = qa_model.encode(row["reference"], convert_to_tensor=True)
    ans_emb = qa_model.encode(row["response"], convert_to_tensor=True)
    score = util.cos_sim(ref_emb, ans_emb).item()
    scores.append(score)

print(f"Skipped {skipped} samples with no reference")
print(f"Valid samples: {len(scores)}/{len(evaluation_dataset)}")
print(f"answer_similarity (generated vs reference): {np.mean(scores):.4f}")


Skipped 0 samples with no reference
Valid samples: 100/100
answer_similarity (generated vs reference): 0.5175
